# Power Plant · проверка компактных моделей на четырёх фолдах
На фолде 0 MGS-KAN выиграл у компактного MLP в 8 из 9 парных сравнений по валидационной RMSE. Здесь проверяем, воспроизводится ли выигрыш на тестовых частях фолдов 1–4. Фолд 0 остаётся разведочным и в итоговую таблицу не входит.

Заранее фиксируем пары размеров MGS-KAN / MLP: **77 / 79**, **125 / 121**, **233 / 229**. Входы — AT, V, AP, RH. Для каждой модели используются те же три пары seed для инициализации и перемешивания. Обучение — 100 эпох Adam, шаг 0,001, батч 64. Лучшая эпоха определяется по валидационной RMSE. Trainer восстанавливает её веса и сохраняет чекпоинт в MLflow. По этим весам один раз вычисляется тестовая RMSE в МВт.

Выбери свободную GPU и выполни **Run All**. При прерывании повтори запуск с той же версией и тем же экспериментом MLflow. Завершённые обучения с тестовой метрикой и чекпоинтом будут переиспользованы.

In [ ]:
PROJECT_DIR = None
DEVICE = "cuda:1"  # Укажи свободную карту в окружении Jupyter.
VERSION = "compact-folds-v1"
GPU_BUDGET_MIB = 4096

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

In [ ]:
from itertools import product
import math

import pandas as pd
from IPython.display import display

from src.config import MonitorSettings, ResourceSettings, TrackingSettings, TrainingSettings
from src.data import dataset_class
from src.experiments import ExperimentRunner
from src.experiments.search import AIRFOIL_STAGES
from src.models import make_model, parameter_count

dataset = "power_plant"
columns = tuple(dataset_class(dataset).columns)
folds = [1, 2, 3, 4]
seeds = [(11, 101), (22, 202), (33, 303)]
models = ["mgs_kan", "mlp"]
training = TrainingSettings(epochs=100, learning_rate=0.001, batch_size=64)
resources = ResourceSettings(_env_file=ENV_FILE, max_parallel_trials=1, cpu_threads=1,
                             gpu_budget_mib=GPU_BUDGET_MIB)
tracking = TrackingSettings(_env_file=ENV_FILE)
monitoring = MonitorSettings(_env_file=ENV_FILE)
output = PROJECT_ROOT / "results" / "power-plant-compact-folds" / VERSION

expected = {"stage-1": (77, 79), "stage-2": (125, 121), "stage-3": (233, 229)}
settings_by_stage = {stage.name: stage.matched_settings(len(columns)) for stage in AIRFOIL_STAGES}
counts = {}
for stage, settings in settings_by_stage.items():
    opts = settings.model_copy(update={"features": len(columns), "init_seed": 0})
    counts[stage] = tuple(parameter_count(make_model(model, opts)) for model in models)
if counts != expected or columns != ("AT", "V", "AP", "RH"):
    raise ValueError("Размеры моделей или порядок признаков отличаются от протокола первого прогона")
display(pd.DataFrame.from_dict(counts, orient="index", columns=models))
print(f"{len(expected) * len(models) * len(folds) * len(seeds)} обучений · {DEVICE} · {output}")

In [ ]:
rows = []
for stage, settings in settings_by_stage.items():
    runner = ExperimentRunner(training=training, resources=resources, tracking=tracking,
                              monitoring=monitoring, model_settings=settings)
    result = runner.run(series_id=f"power-plant-compact-folds-{stage}-{VERSION}",
                        dataset=dataset, feature_columns=columns, models=models, folds=folds,
                        seeds=seeds, device=DEVICE, root=DATA_DIR, evaluation="test")
    for item in result.results:
        trial = item.trial
        rows.append(dict(stage=stage, model=trial.model, parameters=counts[stage][models.index(trial.model)],
                         fold=trial.fold, init_seed=trial.init_seed, shuffle_seed=trial.shuffle_seed,
                         run_id=item.run_id, best_epoch=item.best_epoch,
                         test_rmse=item.test_rmse, test_r2=item.test_r2))
    print(f"{stage}: {len(result.results)} готово, из них {result.reused} взято из MLflow")
runs = pd.DataFrame(rows)

## Воспроизводимость результата
В каждой паре вычитаем тестовую RMSE MLP из тестовой RMSE MGS-KAN на том же фолде и тех же seed. Отрицательная разница означает меньшую ошибку у MGS-KAN. В каждой из 12 ячеек итоговой таблицы — среднее трёх таких разниц в МВт. Сохраняем также абсолютные ошибки и идентификаторы запусков для проверки.

In [ ]:
keys = ["stage", "fold", "init_seed", "shuffle_seed"]
planned = {(stage, fold, init, shuffle, model)
           for stage, fold, (init, shuffle), model in product(expected, folds, seeds, models)}
observed = set(runs[keys + ["model"]].itertuples(index=False, name=None))
if observed != planned or len(runs) != len(planned) or runs.run_id.nunique() != len(planned):
    raise ValueError("Ожидалось 72 уникальных запуска на фолдах 1–4 с запланированными парами seed")
if not runs.test_rmse.map(lambda value: pd.notna(value) and math.isfinite(value) and value >= 0).all():
    raise ValueError("В результатах отсутствует корректная тестовая RMSE")
paired = runs.pivot(index=keys, columns="model", values="test_rmse").rename_axis(columns=None)
paired["delta_rmse_mw"] = paired["mgs_kan"] - paired["mlp"]
fold_deltas = paired["delta_rmse_mw"].groupby(["stage", "fold"]).mean().unstack("fold")
fold_deltas.index = pd.Index(
    [f"{counts[stage][0]} / {counts[stage][1]}" for stage in fold_deltas.index],
    name="Параметры MGS-KAN / MLP",
)
display(fold_deltas.round(4))
output.mkdir(parents=True, exist_ok=True)
runs.to_csv(output / "runs.csv", index=False)
paired.to_csv(output / "paired.csv")
fold_deltas.to_csv(output / "fold-deltas.csv")
print(f"Результаты сохранены в {output}")